# Χαρτογράφηση Περιφερειακού Φορτίου Δικτύου και Διακοπών με το PROC CHART

## Σύνοψη για Στελέχη

Ένας αναλυτής λειτουργιών δικτύου χρησιμοποιεί το PROC CHART για να χαρτογραφήσει ένα συνθετικό δείγμα ενδείξεων μετρητών κυκλωμάτων τροφοδοσίας σε τέσσερις περιοχές εξυπηρέτησης και τέσσερις πηγές παραγωγής. Το σημειωματάριο διατρέχει κατακόρυφα ραβδογράμματα, οριζόντια ραβδογράμματα, γραφήματα μπλοκ και γραφήματα πίτας για να συνοψίσει το μείγμα παραγωγής, να συγκρίνει το μέσο και το συνολικό φορτίο ανά περιοχή, να αναδείξει τη βραδινή αιχμή ζήτησης ανά ώρα, και να κατατάξει τα λεπτά διακοπής ανά πηγή — το είδος της γρήγορης, κειμενικής εξερεύνησης που εκτελεί μια ομάδα ενέργειας και υπηρεσιών κοινής ωφέλειας πριν δεσμευτεί σε μια γραφική αναφορά. Το βήμα DATA ζητά 1,200 γραμμές· αυτό το σημειωματάριο αποδόθηκε σε λειτουργία χωρίς άδεια, η οποία περιορίζει το σύνολο δεδομένων εργασίας στις πρώτες 100 ενδείξεις, οπότε κάθε γράφημα παρακάτω συνοψίζει αυτό το δείγμα των 100 γραμμών.

## Πηγές Δεδομένων

| Σύνολο δεδομένων | Γραμμές | Περιγραφή |
|---------|------|-------------|
| `grid_ops` | 100 (συνθετικό δείγμα) | Μία γραμμή ανά ένδειξη μετρητή κυκλώματος τροφοδοσίας σε ένα περιφερειακό δίκτυο, που δημιουργείται εσωτερικά με `call streaminit(20260531)` και `rand()`. Ο βρόχος του βήματος DATA ζητά 1,200 ενδείξεις, αλλά η λειτουργία χωρίς άδεια περιορίζει το αποθηκευμένο σύνολο δεδομένων σε 100 παρατηρήσεις, οπότε τα γραφήματα παρακάτω περιγράφουν αυτές τις 100 γραμμές. |

# Χαρτογράφηση Περιφερειακού Φορτίου Δικτύου και Διακοπών με το PROC CHART

Το PROC CHART παράγει ραβδογράμματα, γραφήματα μπλοκ και γραφήματα πίτας βασισμένα σε χαρακτήρες απευθείας στο listing — χωρίς να απαιτείται συσκευή ODS Graphics. Αυτό το καθιστά ένα γρήγορο εργαλείο εξερεύνησης πρώτης ματιάς για μια ομάδα λειτουργιών δικτύου που θέλει να *δει* τη μορφή των δεδομένων φορτίου και αξιοπιστίας της πριν κατασκευάσει εκλεπτυσμένα οπτικά GCHART ή SGPLOT.

Σε αυτό το σημειωματάριο:

1. Δημιουργούμε έναν συνθετικό μήνα ενδείξεων μετρητών κυκλωμάτων τροφοδοσίας για έναν περιφερειακό διαχειριστή δικτύου.
2. Χαρτογραφούμε το **μείγμα παραγωγής** (μερίδιο ενδείξεων ανά πηγή).
3. Συγκρίνουμε το **μέσο και συνολικό παραδοθέν φορτίο** μεταξύ των περιοχών εξυπηρέτησης.
4. Αναδεικνύουμε τη **βραδινή αιχμή ζήτησης** ανά ώρα της ημέρας.
5. Χρησιμοποιούμε ένα **γράφημα μπλοκ** για να διασταυρώσουμε την περιοχή με την πηγή παραγωγής.
6. Κατατάσσουμε τα **λεπτά διακοπής** ανά πηγή για να βρούμε την λιγότερο αξιόπιστη τροφοδοσία.

Κάθε εντολή και επιλογή παρακάτω είναι τυπική σύνταξη PROC CHART του SAS 9.4.

## Βήμα 1 — Δημιουργία των συνθετικών δεδομένων λειτουργίας

Το βήμα DATA παρακάτω κατασκευάζει ενδείξεις μετρητών σε έναν βρόχο 1,200 επαναλήψεων. Σε κάθε ένδειξη αποδίδεται μια περιοχή εξυπηρέτησης, μια πηγή παραγωγής (το Gas κυριαρχεί, με τα Wind, Solar και Hydro να συμπληρώνουν το υπόλοιπο) και μια ώρα της ημέρας. Το φορτίο είναι υψηλότερο στο βραδινό παράθυρο 17:00–21:00 (και επισημαίνουμε αυτές τις ενδείξεις ως αιχμή), και αντλούμε τα λεπτά διακοπής από κατανομή Poisson. Το `call streaminit` σταθεροποιεί τον σπόρο τυχαιότητας ώστε τα δεδομένα να είναι αναπαραγώγιμα.

Το NOTE στο log δείχνει το πρακτικό αποτέλεσμα: αυτή η εκτέλεση είναι σε λειτουργία χωρίς άδεια, οπότε το αποθηκευμένο σύνολο δεδομένων `grid_ops` περιορίζεται στις πρώτες 100 από αυτές τις ενδείξεις (100 γραμμές, 7 στήλες). Κάθε γράφημα που ακολουθεί συνοψίζει αυτό το δείγμα των 100 γραμμών, και κάθε log του PROC CHART επιβεβαιώνει ότι διάβασε 100 γραμμές.

In [1]:
/* Synthetic feeder-circuit operations for a regional grid operator */
ΔΕΔΟΜΕΝΑ grid_ops;
    CALL streaminit(20260531);
    LENGTH region $12 source $9 peak_flag $3;
    ARRAY regions[4] $12 _temporary_
        ('North','South','East','West');
    ΕΠΑΝΑΛΗΨΗ meter_id = 1 ΕΩΣ 1200;
        r = ceil(4 * rand('uniform'));
        region = regions[r];
        u = rand('uniform');
        ΕΑΝ u < 0.42 ΤΟΤΕ source = 'Gas';
        ΑΛΛΙΩΣ ΕΑΝ u < 0.70 ΤΟΤΕ source = 'Wind';
        ΑΛΛΙΩΣ ΕΑΝ u < 0.88 ΤΟΤΕ source = 'Solar';
        ΑΛΛΙΩΣ source = 'Hydro';
        hour = floor(24 * rand('uniform'));
        BASE = 18 + 5 * (region = 'North') + 3 * (region = 'West');
        ΕΑΝ hour >= 17 AND hour <= 21 ΤΟΤΕ ΕΠΑΝΑΛΗΨΗ;
            load_mwh = BASE + 12 + 6 * rand('normal');
            peak_flag = 'Yes';
        ΤΕΛΟΣ;
        ΑΛΛΙΩΣ ΕΠΑΝΑΛΗΨΗ;
            load_mwh = BASE + 4 * rand('normal');
            peak_flag = 'No';
        ΤΕΛΟΣ;
        ΕΑΝ load_mwh < 0 ΤΟΤΕ load_mwh = 0;
        outage_min = rand('poisson', 2.5);
        ΕΞΟΔΟΣ;
    ΤΕΛΟΣ;
    ΑΦΑΙΡΕΣΗ r u BASE;
ΕΚΤΕΛΕΣΗ;

NOTE: DATA grid_ops

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote grid_ops (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds


## Βήμα 2 — Μείγμα παραγωγής

Τι μερίδιο ενδείξεων αντιστοιχεί σε κάθε πηγή παραγωγής; Ένα κατακόρυφο ραβδόγραμμα με `TYPE=PERCENT` απαντά απευθείας: τα ύψη των ράβδων είναι το ποσοστό όλων των παρατηρήσεων που εμπίπτουν σε κάθε κατηγορία πηγής. Επειδή η `source` είναι μεταβλητή χαρακτήρων, το PROC CHART αντιμετωπίζει αυτόματα τις τιμές της ως διακριτές κατηγορίες.

In [2]:
ΔΙΑΔΙΚΑΣΙΑ chart ΔΕΔΟΜΕΝΑ=grid_ops;
    VBAR source / type=PERCENT;
ΕΚΤΕΛΕΣΗ;


Percent of SOURCE

         |              *****       
         |              *****       
   40.00 +              *****       
         |              *****       
         |              *****       
   30.00 +              *****       
         |       *****  *****       
         |       *****  *****       
   20.00 +       *****  *****       
         |       *****  *****       
         |*****  *****  *****       
         |*****  *****  *****  *****
   10.00 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |                          
         +--------------------------+
          Hydro  Wind    Gas   Solar
                    SOURCE


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Βήμα 3 — Μέσο παραδοθέν φορτίο ανά περιοχή

Τώρα περνάμε από την καταμέτρηση στη σύνοψη μιας μέτρησης. Ονομάζοντας τη `load_mwh` στο `SUMVAR=` με `TYPE=MEAN`, το μήκος της ράβδου γίνεται το μέσο φορτίο ανά περιοχή, και ζητάμε ρητά τις στήλες στατιστικών: το `MEAN` τυπώνει τον μέσο δίπλα σε κάθε ράβδο και το `CFREQ` προσθέτει μια στήλη αθροιστικής συχνότητας. Η περιοχή North φέρει το υψηλότερο μέσο παραδοθέν φορτίο (μέσος περίπου 28.0 MWh), σε συμφωνία με την περιφερειακή μετατόπιση που ενσωματώθηκε στα δεδομένα, ενώ η South είναι η χαμηλότερη (περίπου 19.8 MWh).

Περνάμε επίσης το `ASCENDING`, με σκοπό να ταξινομήσουμε τις ράβδους από το χαμηλότερο στο υψηλότερο μέσο. Παρατηρήστε στην έξοδο ότι οι ράβδοι **δεν** αναδιατάσσονται — εμφανίζονται με σειρά κατηγορίας (West, South, East, North), με μέσους 24.2, 19.8, 21.7, 28.0 που δεν τρέχουν από τον κοντύτερο στον μακρύτερο. Η αναδιάταξη των ράβδων βάσει του στατιστικού του γραφήματος δεν έχει ακόμη υλοποιηθεί σε αυτή την υλοποίηση του PROC CHART, οπότε τα `ASCENDING`/`DESCENDING` γίνονται αποδεκτά αλλά επί του παρόντος δεν έχουν καμία επίδραση· διαβάστε την κατάταξη από την τυπωμένη στήλη `Mean` αντ' αυτού.

In [3]:
ΔΙΑΔΙΚΑΣΙΑ chart ΔΕΔΟΜΕΝΑ=grid_ops;
    HBAR region / SUMVAR=load_mwh type=mean
                  cfreq mean ascending;
ΕΚΤΕΛΕΣΗ;


Mean of REGION

REGION                                                  Mean           N     Percent
                                                                                    
  West  **********************************             24.17       24.17       23.00
 South  ****************************                   19.80       43.97       21.00
  East  *******************************                21.72       65.69       29.00
 North  ****************************************       28.03       93.72       27.00


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Βήμα 4 — Συνολικό φορτίο ανά περιοχή

Εδώ το `TYPE=SUM` κάνει κάθε ράβδο το *συνολικό* παραδοθέν φορτίο για την περιοχή αντί για τον μέσο, οπότε οι ψηλότερες ράβδοι επισημαίνουν τις περιοχές που παραδίδουν τη μεγαλύτερη συνολική ενέργεια στις δειγματοληπτημένες ενδείξεις. Η North ηγείται και πάλι στο συνολικό παραδοθέν φορτίο.

Η εντολή ζητά επίσης `SUBGROUP=peak_flag`, ζητώντας από το PROC CHART να διαχωρίσει κάθε ράβδο ανάλογα με το αν η ένδειξη έπεσε στο βραδινό παράθυρο αιχμής. Στο SAS αυτό τμηματοποιεί κάθε ράβδο με διακριτό σύμβολο ανά επίπεδο υποομάδας και τυπώνει υπόμνημα. Αυτή η υλοποίηση δεν αποδίδει ακόμη την τμηματοποίηση υποομάδων (ένα τεκμηριωμένο κενό δυνατότητας), οπότε οι ράβδοι βγαίνουν ως συμπαγείς ακολουθίες `*` χωρίς ανάλυση ανά τμήμα — τα *σύνολα* των ράβδων είναι σωστά, αλλά ο διαχωρισμός αιχμής έναντι μη αιχμής δεν εμφανίζεται εδώ. Για να δείτε πόσο φορτίο πέφτει στις ώρες αιχμής, χρησιμοποιήστε την προβολή ώρας-ημέρας στο Βήμα 5.

In [4]:
ΔΙΑΔΙΚΑΣΙΑ chart ΔΕΔΟΜΕΝΑ=grid_ops;
    VBAR region / SUMVAR=load_mwh type=sum
                  SUBGROUP=peak_flag;
ΕΚΤΕΛΕΣΗ;


Sum of REGION

         |                     *****
         |                     *****
         |                     *****
     600 +              *****  *****
         |*****         *****  *****
         |*****         *****  *****
         |*****         *****  *****
     400 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
     200 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |                          
         +--------------------------+
          West   South  East   North
                    REGION


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Βήμα 5 — Μορφή φορτίου κατά τη διάρκεια της ημέρας

Η `hour` είναι συνεχής, οπότε καθορίζουμε εμείς τη διαμέριση με μια ρητή λίστα `MIDPOINTS=` σε κέντρα 4 ωρών (2, 6, 10, 14, 18, 22). Κάθε ράβδος δείχνει το μέσο παραδοθέν φορτίο για τις ενδείξεις κοντά σε εκείνη την ώρα. Η ράβδος με κέντρο στο 18 θα πρέπει να ξεχωρίζει — αυτή είναι η βραδινή αιχμή ζήτησης που ενσωματώσαμε στα δεδομένα.

In [5]:
ΔΙΑΔΙΚΑΣΙΑ chart ΔΕΔΟΜΕΝΑ=grid_ops;
    VBAR hour / SUMVAR=load_mwh type=mean
                midpoints=2 6 10 14 18 22;
ΕΚΤΕΛΕΣΗ;


Mean of HOUR

         |                            *****       
         |                            *****       
   30.00 +                            *****       
         |                            *****  *****
         |                            *****  *****
         |              *****         *****  *****
   20.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
   10.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |                                        
         +----------------------------------------+
            2      6     10     14     18     22  
                            HOUR


NOTE: PROC CHART data=grid_ops


## Βήμα 6 — Περιοχή ανά πηγή παραγωγής (γράφημα μπλοκ)

Ένα γράφημα BLOCK αποδίδει έναν μικρό διμεταβλητό πίνακα ως πλέγμα από μπλοκ. Με `GROUP=source` και `SUMVAR=load_mwh / TYPE=MEAN`, το γράφημα διασταυρώνει κάθε περιοχή με την πηγή παραγωγής που την εξυπηρετεί, με το ύψος του μπλοκ ανάλογο του μέσου φορτίου — ένας συμπαγής τρόπος να εντοπίσετε ποιοι συνδυασμοί περιοχής/πηγής φέρουν το βαρύτερο μέσο φορτίο.

In [6]:
ΔΙΑΔΙΚΑΣΙΑ chart ΔΕΔΟΜΕΝΑ=grid_ops;
    BLOCK region / SUMVAR=load_mwh type=mean
                   GROUP=source;
ΕΚΤΕΛΕΣΗ;


Block chart of Mean of REGION by SOURCE

                          /####\
  /####\                  /####\
  /####\          /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  +----+  +----+  +----+  +----+
   West   South    East   North 
             REGION


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Βήμα 7 — Μείγμα παραγωγής ως γράφημα πίτας

Η ίδια πληροφορία μεριδίου πηγής από το Βήμα 2, σχεδιασμένη ως πίτα. Το PIE με `TYPE=PERCENT` διαστασιολογεί κάθε φέτα ανάλογα με το ποσοστό της επί των συνολικών ενδείξεων και τυπώνει ένα υπόμνημα με τα ποσοστά των φετών δίπλα στο σχήμα.

In [7]:
ΔΙΑΔΙΚΑΣΙΑ chart ΔΕΔΟΜΕΝΑ=grid_ops;
    PIE source / type=PERCENT;
ΕΚΤΕΛΕΣΗ;


Pie chart of SOURCE

          SOURCE     Percent   Percent  Slice
----------------  ----------  --------  ------------------------------
           Hydro       14.00     14.0%  ####
            Wind       28.00     28.0%  ********
             Gas       45.00     45.0%  ++++++++++++++
           Solar       13.00     13.0%  OOOO


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Βήμα 8 — Λεπτά διακοπής ανά πηγή

Τέλος κατατάσσουμε την αξιοπιστία. Το `SUMVAR=outage_min` με `TYPE=SUM` αθροίζει τα λεπτά διακοπής ανά πηγή. Περνάμε `DESCENDING` για να προσπαθήσουμε να ανεβάσουμε την πηγή με τη χειρότερη επίδοση στην κορυφή, αλλά όπως στο Βήμα 3 οι ράβδοι δεν αναδιατάσσονται — τυπώνονται με σειρά κατηγορίας (Hydro, Wind, Gas, Solar) και η αναδιάταξη ράβδων δεν έχει ακόμη υλοποιηθεί. Διαβάστε την κατάταξη από την τυπωμένη στήλη `Sum`: το Gas αντιστοιχεί στα περισσότερα συνολικά λεπτά διακοπής σε αυτό το δείγμα (122), αρκετά μπροστά από το Wind (64), το Solar (43) και το Hydro (38). Αυτό ακολουθεί το μείγμα παραγωγής — το Gas εξυπηρετεί τις περισσότερες ενδείξεις, οπότε συσσωρεύει τα περισσότερα λεπτά διακοπής συνολικά.

In [8]:
ΔΙΑΔΙΚΑΣΙΑ chart ΔΕΔΟΜΕΝΑ=grid_ops;
    HBAR source / SUMVAR=outage_min type=sum DESCENDING;
ΕΚΤΕΛΕΣΗ;


Sum of SOURCE

SOURCE                                                   Sum        Cum.     Percent
                                                                     Sum            
 Hydro  ************                                      38          38       14.00
  Wind  *********************                             64         102       28.00
   Gas  ****************************************         122         224       45.00
 Solar  **************                                    43         267       13.00


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Ερμηνεία των αποτελεσμάτων

Η ανάγνωση των γραφημάτων μαζί δίνει στην ομάδα λειτουργιών μια γρήγορη εικόνα της κατάστασης (επί του δείγματος 100 γραμμών που κατέγραψε αυτή η εκτέλεση):

- **Μείγμα παραγωγής (Βήματα 2 και 7).** Το Gas φέρει το μεγαλύτερο μερίδιο ενδείξεων (περίπου 45%), με το Wind δεύτερο (περίπου 28%) και τα Hydro και Solar να ακολουθούν (περίπου 14% και 13%) — το κατακόρυφο ραβδόγραμμα και η πίτα λένε την ίδια ιστορία με δύο τρόπους, ένας χρήσιμος έλεγχος ορθότητας.
- **Φορτίο ανά περιοχή (Βήματα 3 και 4).** Το HBAR μέσου φορτίου δείχνει την North να έχει το υψηλότερο μέσο παραδοθέν φορτίο (μέσος ~28 MWh) και την South το χαμηλότερο (~20 MWh), σε συμφωνία με την περιφερειακή μετατόπιση που ενσωματώθηκε στα δεδομένα. Το γράφημα SUM επιβεβαιώνει ότι η North παραδίδει επίσης τη μεγαλύτερη συνολική ενέργεια.
- **Ημερήσια μορφή φορτίου (Βήμα 5).** Η ράβδος μεσαίου σημείου με κέντρο στην ώρα 18 είναι σαφώς η ψηλότερη, επιβεβαιώνοντας την αιχμή ζήτησης 17:00–21:00 που ενσωματώσαμε στα δεδομένα — ακριβώς εκεί όπου μια εταιρεία κοινής ωφέλειας θα εστίαζε την απόκριση ζήτησης και τον σχεδιασμό δυναμικότητας.
- **Αξιοπιστία (Βήμα 8).** Το άθροισμα των λεπτών διακοπής ανά πηγή αναδεικνύει το Gas ως τον μεγαλύτερο συντελεστή χρόνου εκτός λειτουργίας σε αυτό το δείγμα (122 λεπτά), τον φυσικό επόμενο στόχο για έλεγχο συντήρησης — αν και αυτό αντικατοπτρίζει κυρίως ότι το Gas εξυπηρετεί τις περισσότερες ενδείξεις.

Δύο επιλογές εμφάνισης που χρησιμοποιήθηκαν εδώ — η αναδιάταξη ράβδων `ASCENDING`/`DESCENDING` (Βήματα 3 και 8) και η τμηματοποίηση ράβδων `SUBGROUP=` (Βήμα 4) — γίνονται αποδεκτές από τον αναλυτή σύνταξης αλλά δεν αποδίδονται ακόμη από αυτή την υλοποίηση του PROC CHART, οπότε οι κατατάξεις και οι διαχωρισμοί διαβάζονται από τις τυπωμένες στήλες στατιστικών παρά από τη σειρά ή τη σκίαση των ράβδων.

Το PROC CHART παράγει μόνο κειμενική έξοδο, οπότε για οπτικά έτοιμα για παρουσίαση σε διοικητικό συμβούλιο η ομάδα θα μετέφερε αυτές τις ίδιες προβολές στο PROC GCHART ή στο PROC SGPLOT. Αλλά ως πρώτη ματιά μηδενικής προετοιμασίας πάνω σε ένα φρέσκο απόσπασμα, αυτά τα γραφήματα απαντούν στα λειτουργικά ερωτήματα — μείγμα, μέγεθος και χρονισμό — σε δευτερόλεπτα.